# UIT-VSFC — CE + completion masking + alpha tại token cuối

Mặc định `ALPHA_SCOPE="terminal_only"`: mỗi nhãn nhận class alpha tại đúng token chứa ký tự cuối của giá trị JSON; các subtoken còn lại và token định dạng giữ trọng số 1. Tra theo **trường + nhãn + vị trí**, không tra bằng token ID toàn cục.

Giữ protocol của bản all-label-token tương ứng: LoRA r=8, LoRA alpha=16, seed=42; MAX_LENGTH=1024; 3 epochs; gamma=0. Cùng parser/collator theo span và cùng khởi tạo được dùng cho hai phạm vi alpha. Đổi `ALPHA_SCOPE="all_label_tokens"` trên runtime mới để chạy nhánh còn lại.

Các mục bên dưới mô tả cả hai phạm vi alpha; cấu hình đang chọn quyết định phạm vi thực tế. Kết quả lịch sử không được khẳng định tái lập chỉ từ việc chạy notebook này.


In [ ]:
# Chỉ cần chạy ô này nếu môi trường chưa có bộ thư viện huấn luyện.
# Nếu đã import các thư viện trước khi cài, restart runtime rồi chạy từ ô kết nối.
# Không cần cài vLLM trong notebook huấn luyện.
%pip install -q transformers==4.46.3 peft==0.10.0 trl==0.8.6 accelerate==0.29.3 datasets==2.21.0 pandas scikit-learn tensorboard


## Kết nối Drive và Hugging Face

Nếu đã import thư viện trước khi cài, restart runtime rồi chạy từ ô này. Đăng nhập tương tác; không ghi token vào notebook.


In [ ]:
from pathlib import Path
from google.colab import drive
from huggingface_hub import notebook_login

if not Path('/content/drive/MyDrive').is_dir():
    mountpoint = Path('/content/drive')
    if mountpoint.exists() and any(mountpoint.iterdir()):
        raise RuntimeError(
            '/content/drive đang chứa file nhưng chưa thấy MyDrive. '
            'Hãy mount Drive ở thư mục trống khác và sửa OUTPUT_DIR tương ứng; '
            'không xóa các file hiện có.'
        )
    drive.mount('/content/drive')

# Đăng nhập tương tác; không ghi token vào notebook.
# Có thể bỏ qua dòng này nếu phiên đã đăng nhập và tải được Vistral.
notebook_login()


## Cấu hình, tokenizer, model và dữ liệu

Chỉnh `SEED`, `ALPHA_SCOPE` và đường dẫn lưu ở đầu ô dưới. Mặc định tạo thư mục run mới dưới `label-focused-tuning/uit_vsfc`, không ghi đè hoặc resume run cũ. Giữ toàn bộ validation và dừng nếu target sẽ bị cắt.


In [ ]:
import os, time, shutil, json, re, hashlib
from itertools import product
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 0. CẤU HÌNH — chỉnh sửa tại đây
# ==========================================
MODEL_ID    = "Viet-Mistral/Vistral-7B-Chat"
SEED        = 42
ALPHA_SCOPE = "terminal_only"  # đổi sang "all_label_tokens" để chạy nhánh toàn bộ token cùng seed
if ALPHA_SCOPE not in {"all_label_tokens", "terminal_only"}:
    raise ValueError("ALPHA_SCOPE chỉ nhận all_label_tokens hoặc terminal_only.")
RUN_NAME    = f"uit_vsfc_mask_alpha_ce_{ALPHA_SCOPE}_r8_len1024_seed{SEED}"
OUTPUT_DIR  = f"/content/drive/MyDrive/label-focused-tuning/uit_vsfc/{RUN_NAME}"
USE_ALPHA   = True
LOSS_NAME   = 'masked_alpha_weighted_ce_no_focal'
SOURCE_NOTEBOOKS_SHA256 = {'uit_vsfc_full_r8_alpha_fixed_len1024.ipynb': '2a07bf62327d8d82413cb0994df23c07e8e2b30cb7930f46428386788de33d98', 'uit_vsfc_mask_alpha_ce_no_focal_r8_len1024.ipynb': '77206f4dd6ba5354ad512045d68d0c13f14b7ec2db4db8ca98c352268f650f7e'}

# LoRA
LORA_R       = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05

# Training
MAX_LENGTH    = 1024
BATCH_SIZE    = 6
GRAD_ACCUM    = 4       # effective batch = 24
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
FOCAL_GAMMA   = 0.0     # CE, không có focal modulation

# Thư mục mới cho từng cấu hình, tránh trộn/ghi đè lần chạy trước.
if os.path.exists(OUTPUT_DIR) and (not os.path.isdir(OUTPUT_DIR) or os.listdir(OUTPUT_DIR)):
    raise FileExistsError(
        f'{OUTPUT_DIR} đã tồn tại và không rỗng. Đổi RUN_NAME/OUTPUT_DIR để chạy mới; '
        'notebook này không tự resume hoặc xóa dữ liệu.'
    )

# Seed đặt trước tải model và được đặt lại ngay trước khởi tạo LoRA.
set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Notebook huấn luyện cần GPU CUDA.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Cấu hình tham chiếu dùng bf16; cần GPU hỗ trợ bf16.")
print(f"Run: {RUN_NAME} | seed={SEED} | alpha scope={ALPHA_SCOPE}")

# ==========================================
# 2. TOKENIZER + MODEL
# ==========================================
print("\n📥 Đang tải Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"   Vocab size: {tokenizer.vocab_size}")

print("\n📥 Đang tải Model (bf16, không quantize)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

allocated = torch.cuda.memory_allocated() / 1e9
print(f"\n✅ Model loaded! VRAM đang dùng: {allocated:.1f} GB")
print(f"   VRAM còn lại: {(torch.cuda.get_device_properties(0).total_memory/1e9 - allocated):.1f} GB")

# ==========================================
# 3. MAPPING LABEL — UIT-VSFC
# ==========================================
sentiment_map = {
    '0': 'Negative', '0.0': 'Negative',
    '1': 'Neutral',  '1.0': 'Neutral',
    '2': 'Positive', '2.0': 'Positive',
}
topic_map = {
    '0': 'Lecturer',   '0.0': 'Lecturer',
    '1': 'Curriculum', '1.0': 'Curriculum',
    '2': 'Facility',   '2.0': 'Facility',
    '3': 'Others',     '3.0': 'Others',
}
SENTIMENT_LABELS = ["Negative", "Neutral", "Positive"]
TOPIC_LABELS     = ["Lecturer", "Curriculum", "Facility", "Others"]

# ==========================================
# 4. SYSTEM PROMPT — chỉnh nhãn UIT-VSFC
# ==========================================
SYSTEM_PROMPT = (
    "Thực hiện đồng thời hai nhiệm vụ:\n"
    "(1) Cảm xúc (Sentiment): Negative, Neutral, Positive.\n"
    "(2) Chủ đề (Topic): Lecturer, Curriculum, Facility, Others.\n"
    "Chỉ trả về JSON với đúng hai khóa: Sentiment và Topic. Không giải thích thêm."
)
FEW_SHOT_EXAMPLE = """\
Ví dụ:
Phản hồi: "giảng viên giảng dạy nhiệt tình , có đầy đủ các hoạt động để sinh viên rèn luyện và học tập ."
Output: {"Sentiment": "Positive", "Topic": "Lecturer"}"""

# ==========================================
# 5. PROMPT BUILDER
# ==========================================
def generate_train_prompt(row) -> str:
    sent_label  = sentiment_map.get(str(row['sentiment']).strip(), "Neutral")
    topic_label = topic_map.get(str(row['topic']).strip(), "Others")
    target_json = f'{{"Sentiment": "{sent_label}", "Topic": "{topic_label}"}}'
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"{FEW_SHOT_EXAMPLE}\n\nPhản hồi: \"{row['sentence']}\""},
        {"role": "assistant", "content": target_json},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

print("📂 Đang load dữ liệu UIT-VSFC từ Parquet...")
data_files = {
    "train": "hf://datasets/uitnlp/vietnamese_students_feedback@~parquet/default/train/0000.parquet",
    "validation": "hf://datasets/uitnlp/vietnamese_students_feedback@~parquet/default/validation/0000.parquet"
}
ds = load_dataset("parquet", data_files=data_files)

df_train = ds["train"].to_pandas()
df_val   = ds["validation"].to_pandas()

print(f"   Train raw : {len(df_train)}")
print(f"   Val raw   : {len(df_val)}")
print(f"   Columns   : {df_train.columns.tolist()}")
print(f"\n   Phân phối Sentiment (train):\n{df_train['sentiment'].value_counts().to_string()}")
print(f"\n   Phân phối Topic (train):\n{df_train['topic'].value_counts().to_string()}")

def validate_frame(df, split):
    required = {'sentence', 'sentiment', 'topic'}
    if not required.issubset(df.columns):
        raise ValueError(f'{split}: thiếu cột {sorted(required - set(df.columns))}')
    if len(df) == 0:
        raise ValueError(f'{split}: dữ liệu rỗng')
    for col, mapping in [('sentiment', sentiment_map), ('topic', topic_map)]:
        values = df[col].map(lambda value: str(value).strip())
        bad = ~values.isin(mapping)
        if bad.any():
            raise ValueError(f'{split}/{col}: nhãn không hợp lệ ở {df.index[bad].tolist()[:10]}')
    valid_text = df['sentence'].map(lambda value: isinstance(value, str) and bool(value.strip()))
    if not valid_text.all():
        raise ValueError(f'{split}: sentence rỗng/không phải chuỗi ở {df.index[~valid_text].tolist()[:10]}')

validate_frame(df_train, 'train')
validate_frame(df_val, 'validation')

# ==========================================
# 7. FILTER THEO ĐỘ DÀI + TẠO DATASET
# ==========================================
all_prompts_train = df_train.apply(generate_train_prompt, axis=1).tolist()
lengths = [len(ids) for ids in tokenizer(all_prompts_train)["input_ids"]]
df_train["prompt_len"] = lengths
df_train_filtered = df_train[df_train["prompt_len"] <= MAX_LENGTH].copy()
print(f"\n   Sau filter (≤{MAX_LENGTH} tokens): {len(df_train_filtered)}/{len(df_train)} mẫu train")

prompts_train = pd.DataFrame(
    df_train_filtered.apply(generate_train_prompt, axis=1), columns=["text"]
)
prompts_val = pd.DataFrame(
    df_val.apply(generate_train_prompt, axis=1), columns=["text"]
)
train_dataset = Dataset.from_pandas(prompts_train, preserve_index=False)
eval_dataset  = Dataset.from_pandas(prompts_val,   preserve_index=False)
print(f"✅ Dataset sẵn sàng — Train: {len(train_dataset)} | Val: {len(eval_dataset)}")

# Validation được giữ đủ; dừng nếu SFTTrainer sẽ cắt câu trả lời.
val_prompt_lengths = [len(ids) for ids in tokenizer(prompts_val['text'].tolist())['input_ids']]
too_long_val = [i for i, length in enumerate(val_prompt_lengths) if length > MAX_LENGTH]
if too_long_val:
    raise ValueError(
        f'{len(too_long_val)} mẫu validation vượt MAX_LENGTH={MAX_LENGTH}; '
        f'ví dụ vị trí {too_long_val[:10]}. Không tự loại mẫu hoặc đổi max length riêng cho ablation.'
    )
if len(train_dataset) == 0:
    raise ValueError('Không còn mẫu train sau filter.')


## Alpha theo lớp và masked weighted CE

Tính riêng $α^{(q)}_c=N/(K_q n^{(q)}_c)$ từ **filtered train** cho mỗi nhiệm vụ $q$.

$$L=\frac{1}{|M|}\sum_{t\in M} a_t\,[-\log p(y_t\mid x,y_{<t})].$$

`M` là các token completion còn lại sau mask và causal shift. Với `all_label_tokens`, $a_t$ bằng alpha của trường/nhãn tương ứng trên **mọi token giao vùng giá trị nhãn**; các vị trí khác bằng 1. Không chia theo số token nhãn hoặc tổng alpha. Nếu tokenizer gộp nhãn với dấu câu trong cùng token thì trọng số áp dụng cho token đó; loss không thể chia nhỏ hơn một token.

`LoRA alpha=16` là hệ số adapter riêng, không phải trọng số lớp. `gamma=0` được cố định vì đây là thí nghiệm CE.


In [ ]:
def compute_sample_weights(df, col, label_map: dict) -> dict:
    raw = df[col].astype(str).str.strip()
    invalid = ~raw.isin(label_map)
    if invalid.any():
        raise ValueError(f"{col}: nhãn không hợp lệ: {sorted(raw[invalid].unique())}")
    names = raw.map(label_map).tolist()
    if not names:
        raise ValueError(f"{col}: không có mẫu để tính alpha.")
    classes = sorted(set(names))
    values = compute_class_weight("balanced", classes=np.array(classes), y=names)
    result = {name: float(value) for name, value in zip(classes, values)}
    print(f"  [{col}] weights: { {k: round(v, 3) for k, v in result.items()} }")
    return result


print("\n📊 Tính class weights từ filtered train...")
sent_weights = compute_sample_weights(df_train_filtered, "sentiment", sentiment_map)
topic_weights = compute_sample_weights(df_train_filtered, "topic", topic_map)
for field, names, weights in [
    ("Sentiment", SENTIMENT_LABELS, sent_weights),
    ("Topic", TOPIC_LABELS, topic_weights),
]:
    missing = set(names) - set(weights)
    if missing:
        raise ValueError(f"Train thiếu lớp {field}: {sorted(missing)}")
if not USE_ALPHA or FOCAL_GAMMA != 0:
    raise ValueError("Thí nghiệm này yêu cầu USE_ALPHA=True và FOCAL_GAMMA=0.")


def masked_alpha_ce_loss(logits, labels, alpha_weights):
    """Mean weighted CE trên các shifted completion token, giữ mẫu số gốc."""
    if labels.shape != alpha_weights.shape or logits.shape[:2] != labels.shape:
        raise ValueError("Shape logits/labels/alpha_weights không khớp.")
    shifted_logits = logits[..., :-1, :].contiguous().reshape(-1, logits.size(-1))
    targets = labels[..., 1:].contiguous().reshape(-1).to(logits.device)
    weights = alpha_weights[..., 1:].contiguous().reshape(-1).to(
        device=logits.device, dtype=torch.float32
    )
    keep = targets != -100
    if not keep.any().item():
        raise ValueError("Batch không có shifted completion token để tính loss.")
    ce = F.cross_entropy(shifted_logits[keep].float(), targets[keep], reduction="none")
    return (weights[keep] * ce).mean()


class MaskedAlphaCETrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Custom loss dùng mean theo microbatch, không tiêu thụ num_items_in_batch.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        model_inputs = dict(inputs)
        alpha = model_inputs.pop("alpha_weights", None)
        if alpha is None:
            raise ValueError("Thiếu alpha_weights từ TaskAwareAlphaCollator.")
        labels = model_inputs.pop("labels")
        # Không tính thêm loss mặc định của model vì đã có custom CE bên dưới.
        outputs = model(**model_inputs)
        loss = masked_alpha_ce_loss(outputs.logits, labels, alpha)
        outputs["loss"] = loss
        return (loss, outputs) if return_outputs else loss


# Kiểm tra causal shift, mask và mẫu số trên tensor nhỏ trước khi train.
def check_loss_formula():
    generator = torch.Generator().manual_seed(123)
    logits = torch.randn(2, 5, 7, generator=generator, requires_grad=True)
    labels = torch.tensor([[-100, -100, 2, 3, 4], [-100, 1, 5, -100, -100]])
    alpha = torch.ones_like(labels, dtype=torch.float32)
    alpha[0, 2:4] = 3.0
    alpha[1, 1:3] = 0.5
    keep = labels[:, 1:] != -100
    ce = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, 7), labels[:, 1:].reshape(-1),
        ignore_index=-100, reduction="none",
    ).reshape_as(keep)
    expected = (alpha[:, 1:] * ce)[keep].sum() / keep.sum()
    actual = masked_alpha_ce_loss(logits, labels, alpha)
    assert torch.allclose(actual, expected, atol=1e-6)
    plain = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, 7), labels[:, 1:].reshape(-1), ignore_index=-100
    )
    assert torch.allclose(masked_alpha_ce_loss(logits, labels, torch.ones_like(alpha)), plain)
    actual.backward()
    assert torch.isfinite(logits.grad).all()
    assert torch.count_nonzero(logits.grad[0, 0]) == 0  # target index 1 bị mask
    assert torch.count_nonzero(logits.grad[:, -1]) == 0  # logit cuối bị shift bỏ
    print("✅ Công thức CE: alpha=1 khớp CE gốc; shift, mask và gradient đúng.")


check_loss_formula()


## Xác định toàn bộ token của nhãn và tạo collator

Đọc JSON từ chính các token completion mà collator gốc giữ lại. Xác định khoảng ký tự của từng giá trị nhãn, rồi lấy tất cả token giao với khoảng đó. Không tokenize nhãn đứng riêng để lookup alpha; không phụ thuộc token cuối có ID riêng hay không.


In [ ]:
def find_completion_label_spans(active_ids):
    """Trả về JSON và các chỉ số token tương đối phủ từng giá trị nhãn."""
    ids = tuple(int(token_id) for token_id in active_ids)
    if not ids:
        raise ValueError("Completion không có token được học.")
    decode_kwargs = dict(skip_special_tokens=False, clean_up_tokenization_spaces=False)
    decoded = tokenizer.decode(ids, **decode_kwargs)
    start = decoded.find("{")
    if start < 0 or decoded[:start].strip():
        raise ValueError("Completion sau mask phải bắt đầu bằng target JSON.")
    try:
        obj, end = json.JSONDecoder().raw_decode(decoded, start)
    except json.JSONDecodeError as exc:
        raise ValueError("Target JSON thiếu/cắt; kiểm tra MAX_LENGTH và completion mask.") from exc
    if not isinstance(obj, dict) or set(obj) != {"Sentiment", "Topic"}:
        raise ValueError(f"Target JSON phải có đúng hai khóa Sentiment và Topic: {obj}")

    # Prefix decode giúp dùng đúng token trong ngữ cảnh completion thực tế.
    boundaries = [0]
    for stop in range(1, len(ids) + 1):
        prefix = tokenizer.decode(ids[:stop], **decode_kwargs)
        if not decoded.startswith(prefix) or len(prefix) < boundaries[-1]:
            raise ValueError("Prefix decode không ổn định; không thể xác định span an toàn.")
        boundaries.append(len(prefix))

    spans = {}
    char_spans = {}
    pattern = r'"(?P<field>Sentiment|Topic)"\s*:\s*"(?P<label>[^"\\]*)"'
    for match in re.finditer(pattern, decoded[start:end]):
        field, name = match.group("field"), match.group("label")
        if field in spans or not name or obj[field] != name:
            raise ValueError("Không xác định được duy nhất giá trị nhãn hợp lệ.")
        left = start + match.start("label")
        right = start + match.end("label")
        positions = tuple(
            i for i in range(len(ids))
            if boundaries[i + 1] > boundaries[i]
            and boundaries[i + 1] > left and boundaries[i] < right
        )
        if not positions:
            raise ValueError(f"Không tìm được token cho {field}={name}.")
        if positions != tuple(range(positions[0], positions[-1] + 1)):
            raise ValueError(f"Span token không liên tục: {field}={name}.")
        spans[field] = positions
        char_spans[field] = (left, right)
    if set(spans) != {"Sentiment", "Topic"}:
        raise ValueError("Không tìm được đủ hai vùng nhãn.")
    if set(spans["Sentiment"]) & set(spans["Topic"]):
        raise ValueError("Hai trường chung một token; không thể áp hai alpha độc lập.")
    return obj, spans, char_spans


class TaskAwareAlphaCollator:
    def __init__(self, inner, sentiment_weights, topic_weights, scope="all_label_tokens"):
        if scope not in {"all_label_tokens", "terminal_only"}:
            raise ValueError(f"Phạm vi alpha không hợp lệ: {scope}")
        self.inner = inner
        self.scope = scope
        self.weights_by_field = {
            "Sentiment": {k: float(v) for k, v in sentiment_weights.items()},
            "Topic": {k: float(v) for k, v in topic_weights.items()},
        }
        self._cache = {}

    def __call__(self, features):
        batch = self.inner(features)
        labels = batch["labels"]
        alpha = torch.ones_like(labels, dtype=torch.float32)
        for row in range(labels.size(0)):
            active_idx = torch.where(labels[row] != -100)[0]
            if active_idx.numel() == 0:
                raise ValueError("Mẫu không có completion token được học.")
            # Notebook này không packing; mỗi hàng chứa một completion liên tục.
            if active_idx.numel() > 1 and not torch.all(active_idx[1:] - active_idx[:-1] == 1):
                raise ValueError("Completion mask không liên tục; không hỗ trợ packing nhiều mẫu.")
            ids = tuple(labels[row, active_idx].tolist())
            if ids not in self._cache:
                self._cache[ids] = find_completion_label_spans(ids)
            obj, spans, _ = self._cache[ids]
            for field, class_weights in self.weights_by_field.items():
                name = obj[field]
                if name not in class_weights:
                    raise ValueError(f"Thiếu alpha của {field}={name}.")
                relative = spans[field] if self.scope == "all_label_tokens" else spans[field][-1:]
                absolute = active_idx[list(relative)]
                alpha[row, absolute] = class_weights[name]
        batch["alpha_weights"] = alpha
        return batch


response_template_ids = tokenizer.encode("[/INST]", add_special_tokens=False)
completion_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids, tokenizer=tokenizer, mlm=False,
)
collator = TaskAwareAlphaCollator(
    completion_collator, sent_weights, topic_weights, scope=ALPHA_SCOPE,
)


## Kiểm tra đủ 12 cặp nhãn trước khi train

Xác nhận mask không đổi; mọi token nhãn nhận đúng alpha của nhiệm vụ; các token khác giữ 1. Với fast tokenizer, đối chiếu thêm với `offset_mapping` trên **toàn bộ prompt** như một kiểm tra độc lập. In bảng token thực trong JSON, lưu chi tiết vào manifest và kiểm tra batch có padding.


In [ ]:
alpha_audit = []
token_rows = {}
probe_features = []
probe_source = df_train_filtered.loc[df_train_filtered["prompt_len"].idxmin()].copy()
use_offset_audit = bool(getattr(tokenizer, "is_fast", False))

for sent, topic in product(SENTIMENT_LABELS, TOPIC_LABELS):
    probe = probe_source.copy()
    probe["sentiment"] = next(k for k, v in sentiment_map.items() if v == sent)
    probe["topic"] = next(k for k, v in topic_map.items() if v == topic)
    prompt = generate_train_prompt(probe)
    encoded = tokenizer(prompt, truncation=False, return_offsets_mapping=use_offset_audit)
    offsets = encoded.pop("offset_mapping", None)
    if len(encoded["input_ids"]) > MAX_LENGTH:
        raise ValueError("Prompt audit vượt MAX_LENGTH.")
    features = {k: encoded[k] for k in ("input_ids", "attention_mask")}
    probe_features.append(features)
    original = completion_collator([features])
    batch = collator([features])
    if not torch.equal(batch["labels"], original["labels"]):
        raise ValueError("Mask completion đã bị thay đổi.")

    active = torch.where(batch["labels"][0] != -100)[0]
    active_ids = batch["labels"][0, active].tolist()
    obj, spans, char_spans = find_completion_label_spans(active_ids)
    if obj != {"Sentiment": sent, "Topic": topic}:
        raise ValueError(f"Completion audit sai nhãn: {obj}")

    expected = torch.ones_like(batch["alpha_weights"][0])
    field_audit = {}
    target_json = f'{{"Sentiment": "{sent}", "Topic": "{topic}"}}'
    target_start = prompt.rfind(target_json)
    if target_start < 0:
        raise ValueError("Không tìm thấy target JSON trong prompt audit.")
    for field, class_weights in [("Sentiment", sent_weights), ("Topic", topic_weights)]:
        positions = list(spans[field])
        absolute = active[positions].tolist()
        name = obj[field]
        if use_offset_audit:
            # Xác định khoảng nhãn từ target đã biết, độc lập với prefix parser.
            marker = f'"{field}": "'
            left = target_start + target_json.index(marker) + len(marker)
            right = left + len(name)
            by_offset = [i for i, (a, b) in enumerate(offsets) if b > a and b > left and a < right]
            if absolute != by_offset:
                raise ValueError(f"Span {field} khác offset_mapping: {absolute} != {by_offset}")
        selected = absolute if ALPHA_SCOPE == "all_label_tokens" else absolute[-1:]
        expected[selected] = float(class_weights[name])
        ids = [active_ids[i] for i in positions]
        pieces = tokenizer.convert_ids_to_tokens(ids)
        decoded_span = tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
        if name not in decoded_span:
            raise ValueError(f"Span decode không phủ đủ nhãn {field}={name}: {decoded_span!r}")
        token_rows[(field, name)] = {
            "field": field, "label": name, "n_tokens_in_json": len(ids),
            "tokens": " | ".join(pieces), "alpha": float(class_weights[name]),
            "n_alpha_positions": len(selected),
        }
        field_audit[field] = {
            "label": name, "relative_positions": positions,
            "absolute_positions": absolute, "token_ids": ids, "tokens": pieces,
            "weighted_absolute_positions": selected, "alpha": float(class_weights[name]),
        }
    if not torch.allclose(batch["alpha_weights"][0], expected, rtol=0, atol=1e-6):
        raise ValueError(f"Alpha sai ở cặp ({sent}, {topic}).")
    alpha_audit.append({"Sentiment": sent, "Topic": topic, "fields": field_audit})

# Kiểm tra cả batch nhiều độ dài: prompt/padding vẫn -100, alpha padding vẫn 1.
original_padded = completion_collator(probe_features)
padded = collator(probe_features)
assert torch.equal(padded["labels"], original_padded["labels"])
padding = padded["attention_mask"] == 0
assert torch.all(padded["labels"][padding] == -100)
assert torch.all(padded["alpha_weights"][padding] == 1)
assert len(alpha_audit) == len(SENTIMENT_LABELS) * len(TOPIC_LABELS) == 12

print(pd.DataFrame(token_rows.values()).to_string(index=False))
print(f"✅ Đủ {len(alpha_audit)}/12 cặp; mask và alpha đúng; scope={ALPHA_SCOPE}.")
print(f"   Kiểm tra offset độc lập: {use_offset_audit}; gamma={FOCAL_GAMMA}; seed={SEED}")


## LoRA, thiết lập train và lưu cấu hình thực nghiệm

Giữ rank/hyperparameter và tiêu chí chọn best theo validation loss như bản tham chiếu. `label_names=["labels"]` bảo đảm PEFT evaluation dùng custom loss. Seed được đặt lại ngay trước tạo LoRA, đồng thời truyền vào trainer và data loader.

Các run cũ chưa đặt seed trước tạo LoRA không tự trở thành đối chứng cùng khởi tạo. Để có đối chứng mới cùng protocol, đổi `ALPHA_SCOPE="terminal_only"` và chạy từ đầu trên runtime mới, giữ nguyên seed/cấu hình.


In [ ]:
# ==========================================
# 10. LORA CONFIG
# ==========================================
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
set_seed(SEED)  # Cố định khởi tạo LoRA, không chỉ seed của Trainer.
model_with_lora = get_peft_model(model, peft_config)
trainable, total = 0, 0
for p in model_with_lora.parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()
print(f"✅ LoRA config: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Trainable params : {trainable:,} ({100*trainable/total:.2f}%)")

# ==========================================
# 12. TRAINING ARGS
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=SEED,
    data_seed=SEED,
    label_names=["labels"],
    prediction_loss_only=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim="adamw_torch_fused",
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    report_to="tensorboard",
    group_by_length=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

# Lưu thông tin chạy trước train; không dùng test để chọn checkpoint.
import importlib.metadata

def frame_sha256(df):
    payload = df[['sentence', 'sentiment', 'topic']].to_json(orient='split', force_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

manifest = {
    'run_name': RUN_NAME,
    'status': 'ready',
    'model_id': MODEL_ID,
    'notebook': 'uit_vsfc_mask_alpha_ce_terminal_only_r8_len1024.ipynb',
    'source_notebooks_sha256': SOURCE_NOTEBOOKS_SHA256,
    'seed': SEED,
    'model_resolved_revision': getattr(model.config, '_commit_hash', None),
    'tokenizer_resolved_revision': tokenizer.init_kwargs.get('_commit_hash'),
    'tokenizer_backend_sha256': (
        hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode('utf-8')).hexdigest()
        if getattr(tokenizer, 'is_fast', False) else None
    ),
    'loss': {
        'name': LOSS_NAME,
        'completion_mask': True,
        'gamma': FOCAL_GAMMA,
        'alpha_balancing': USE_ALPHA,
        'alpha_lookup': 'json_field_and_actual_completion_span',
        'alpha_scope': ALPHA_SCOPE,
        'outside_label_alpha': 1.0,
        'label_length_normalization': False,
        'offset_audit': use_offset_audit,
        'reduction': 'mean_over_unmasked_shifted_response_tokens',
    },
    'lora': {'r': LORA_R, 'lora_alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT},
    'max_length': MAX_LENGTH,
    'train_rows_raw': len(df_train),
    'train_rows_filtered': len(df_train_filtered),
    'validation_rows': len(df_val),
    'train_filtered_sha256': frame_sha256(df_train_filtered),
    'validation_sha256': frame_sha256(df_val),
    'data_files': data_files,
    'sentiment_weights_from_train': {k: float(v) for k, v in sent_weights.items()},
    'topic_weights_from_train': {k: float(v) for k, v in topic_weights.items()},
    'alpha_audit_12_pairs': alpha_audit,
    'label_token_audit': list(token_rows.values()),
    'system_prompt': SYSTEM_PROMPT,
    'few_shot_example': FEW_SHOT_EXAMPLE,
    'training_arguments': json.loads(training_args.to_json_string()),
    'seed_note': 'set_seed(SEED) trước tải model và ngay trước tạo LoRA; Trainer seed/data_seed cùng SEED. Không bảo đảm các run lịch sử có cùng khởi tạo.',
    'runtime_versions': {
        name: importlib.metadata.version(name)
        for name in ['torch', 'transformers', 'tokenizers', 'peft', 'trl', 'accelerate', 'datasets', 'scikit-learn']
    },
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)
print(f'📁 Run riêng: {OUTPUT_DIR}')
print('Checkpoint được chọn bằng validation loss của chính objective này, không phải test MF1.')


## Huấn luyện và lưu adapter

Chạy một cấu hình đang chọn, không resume tự động. Lưu `best_model`, `second_best_model` nếu checkpoint còn tồn tại, và `last_epoch_model` theo checkpoint được lưu cuối; kiểm tra đường dẫn thực tế trong manifest. `save_total_limit=3` giữ cơ chế rotation của bản tham chiếu.


In [ ]:
# ==========================================
# 13. TRAINER & TRAINING
# ==========================================
trainer = MaskedAlphaCETrainer(
    model=model_with_lora,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_LENGTH,
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
            early_stopping_threshold=0.001,
        )
    ],
)

allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM trước train: {allocated:.1f} GB")
print(f"\n🚀 CE + masking + alpha; scope={ALPHA_SCOPE}; seed={SEED}")
print(f"   Tổng steps dự kiến: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS}")

start_time = time.time()
trainer.train()
total_time = time.time() - start_time
print(f"\n🎉 Huấn luyện xong! Thời gian: {total_time/3600:.1f} giờ ({total_time/60:.0f} phút)")

# ==========================================
# 14. LƯU MODEL — chỉ báo cáo thư mục thực sự tồn tại
# ==========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_state()
eval_logs = [log for log in trainer.state.log_history if 'eval_loss' in log]
sorted_eval_logs = sorted(eval_logs, key=lambda item: item['eval_loss'])
saved_models = {}

# load_best_model_at_end=True: model hiện tại là best theo validation loss.
best_dir = os.path.join(OUTPUT_DIR, 'best_model')
trainer.model.save_pretrained(best_dir)
tokenizer.save_pretrained(best_dir)
saved_models['best_model'] = best_dir
print(f'🥇 Best model: {best_dir}')
print(f'   Checkpoint được trainer chọn: {trainer.state.best_model_checkpoint}')
print(f'   Best validation loss: {trainer.state.best_metric}')

if len(sorted_eval_logs) >= 2:
    second = sorted_eval_logs[1]
    second_src = os.path.join(OUTPUT_DIR, f"checkpoint-{int(second['step'])}")
    second_dir = os.path.join(OUTPUT_DIR, 'second_best_model')
    if os.path.isdir(second_src):
        shutil.copytree(second_src, second_dir)
        tokenizer.save_pretrained(second_dir)
        saved_models['second_best_model'] = second_dir
        print(f"🥈 Second best: {second_dir} (step {int(second['step'])})")
    else:
        print('ℹ️ Second-best checkpoint đã bị rotation xóa; không tạo thư mục second_best_model.')

available_ckpts = [
    name for name in os.listdir(OUTPUT_DIR)
    if re.fullmatch(r'checkpoint-\d+', name) and os.path.isdir(os.path.join(OUTPUT_DIR, name))
]
last_saved_step = None
if available_ckpts:
    last_saved_step = max(int(name.split('-')[-1]) for name in available_ckpts)
    last_src = os.path.join(OUTPUT_DIR, f'checkpoint-{last_saved_step}')
    # Giữ tên cũ để inference thuận tiện. Đây là checkpoint LƯU CUỐI,
    # có thể chưa phải bước kết thúc epoch/cuối training.
    last_dir = os.path.join(OUTPUT_DIR, 'last_epoch_model')
    shutil.copytree(last_src, last_dir)
    tokenizer.save_pretrained(last_dir)
    saved_models['last_saved_checkpoint'] = last_dir
    print(f'📌 Checkpoint lưu cuối: {last_dir} (step {last_saved_step})')

manifest.update({
    'status': 'completed',
    'train_seconds': total_time,
    'final_global_step': int(trainer.state.global_step),
    'best_model_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_loss': trainer.state.best_metric,
    'last_saved_checkpoint_step': last_saved_step,
    'saved_models': saved_models,
    'eval_logs_sorted_by_objective_loss': sorted_eval_logs,
})
with open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

print('\n📊 Validation loss theo bước (chỉ so trong cấu hình này):')
for log in sorted_eval_logs:
    print(f"   Step {int(log['step']):>5} | eval_loss = {log['eval_loss']:.4f}")
print('\n📁 Các adapter thực sự đã lưu:')
for name, folder in saved_models.items():
    print(f'   {name}: {folder}')
print('📄 trainer_state.json và run_manifest.json đã lưu trong OUTPUT_DIR.')


## Đánh giá và đối chiếu sau train

Dùng pipeline inference UIT-VSFC đang dùng cho đối chứng; đổi `ADAPTER_PATH` sang adapter của run này. Giữ cùng prompt, decoder, split, tiêu chí chọn checkpoint và cách xử lý lỗi parse. So macro-F1 Sentiment/Topic, F1 từng lớp (đặc biệt Neutral, Facility, Others) và tỷ lệ JSON hợp lệ. Chọn cấu hình bằng validation, giữ test cho đánh giá cuối.

`eval_loss` được dùng để chọn best bên trong mỗi run như bản tham chiếu; không dùng trị số loss giữa hai phạm vi alpha để kết luận chất lượng phân loại. Notebook không chia loss nhãn theo độ dài và không chia tổng alpha, để phép thử chỉ đổi phạm vi gán alpha.

Manifest lưu nguồn notebook, seed, phạm vi alpha, hash dữ liệu/tokenizer, class weights, token nhãn trong JSON, audit đủ 12 cặp, cấu hình và phiên bản thư viện. Chạy notebook lần đầu cần runtime mới và thư mục đầu ra chưa có dữ liệu.
